In [ ]:
import math
import re
import numpy as np
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
doc1 = " ".join([
    "Stanford University is a great place for research. Many students aspire to study here.",
    "California is known for its beautiful weather and top-ranked universities.",
    "The University of California, Berkeley, is also a highly reputed institution.",
    "MIT is one of the most prestigious universities in the world.",
    "Harvard University has a long history of excellence in education and research."
])

doc2 = " ".join([
    "Artificial Intelligence is transforming industries worldwide.",
    "Machine Learning and Deep Learning are subsets of AI.",
    "The impact of AI on the healthcare sector is revolutionary.",
    "AI technologies are being used in autonomous vehicles.",
    "Ethical concerns around AI are growing as the technology advances."
])

doc3 = "The University of California, Berkeley, is also a highly reputed institution."
doc4 = "MIT is one of the most prestigious universities in the world."
doc5 = "Harvard University has a long history of excellence in education and research."

documents = [doc1, doc2, doc3, doc4, doc5]

query = "top universities in California"

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    return [stemmer.stem(word) for word in tokens if word not in stop_words]

processed_docs = [preprocess(doc) for doc in documents]
processed_query = preprocess(query)

In [ ]:
vocabulary = sorted(set(word for doc in processed_docs for word in doc))

In [ ]:
N = len(processed_docs)
idf = {}

for word in vocabulary:
    df = sum(1 for doc in processed_docs if word in doc)
    idf[word] = math.log((N + 1) / (df + 1)) + 1

In [ ]:
def compute_tfidf_vector(words):
    tf = Counter(words)
    total_words = len(words)

    vector = []
    for word in vocabulary:
        tf_value = tf[word] / total_words if word in tf else 0
        vector.append(tf_value * idf[word])

    return np.array(vector)

doc_vectors = [compute_tfidf_vector(doc) for doc in processed_docs]
query_vector = compute_tfidf_vector(processed_query)

In [ ]:
def cosine_similarity(vec1, vec2):
    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0
    return dot / (norm1 * norm2)

scores = []

for i, doc_vector in enumerate(doc_vectors):
    similarity = cosine_similarity(doc_vector, query_vector)
    scores.append((i + 1, similarity))

ranked = sorted(scores, key=lambda x: x[1], reverse=True)

In [ ]:
print("\nRanking \n")

for doc_id, score in ranked:
    print(f"Document {doc_id} → Similarity: {score:.5f}")
    print("-" * 60)


Ranking 

Document 1 → Similarity: 0.50908
------------------------------------------------------------
Document 3 → Similarity: 0.47885
------------------------------------------------------------
Document 4 → Similarity: 0.18872
------------------------------------------------------------
Document 5 → Similarity: 0.15696
------------------------------------------------------------
Document 2 → Similarity: 0.00000
------------------------------------------------------------
